# Backbone benchmarking -- Kaggle runner

Thin wrapper. All logic lives in the `bbeval` package; this notebook only
installs it, points it at the mounted datasets, and runs it.

Attach **MVTec AD** and **VisA** as datasets and enable a **GPU** accelerator.


## 1. Install

In [ ]:
%pip install -q -e /kaggle/working/backbone-eval[siglip2]
# CLIP is a git dependency; skip this line for a SigLIP2-only run.
%pip install -q "git+https://github.com/openai/CLIP.git"


## 2. Settings

`siglip2_dense_readout` is the variable under test:

* `"map_token"` -- pool each patch through SigLIP2's own attention-pooling head
* `"raw"` -- leave trunk tokens unprojected; the CLIP-shaped control that
  reproduces the published chance-level localisation


In [ ]:
from bbeval import BackboneEvalConfig, run_evaluation

config = BackboneEvalConfig(
    mvtec_root="/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection",
    visa_root="/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922",
    output_root="/kaggle/working/results",
    weights_dir="/kaggle/working/weights",
    backbones=("clip", "siglip2"),
    siglip2_dense_readout="map_token",
    corruptions_enabled=False,      # clean-only for the backbone comparison
    device="cuda",
)
print(config.fingerprint())


## 3. Smoke test

A few images per category first. Nothing is saved when `limit` is set, so this
cannot pollute the real artefacts.


In [ ]:
from dataclasses import replace
from bbeval.engine import load_backbones

smoke = replace(config, limit=4, categories={"mvtec": ("hazelnut",), "visa": ("candle",)})
backbones = load_backbones(smoke)


## 4. Run

Writes low-resolution anomaly maps, raw scores, ground truth, a run manifest
recording every per-backbone choice, and the metric tables.


In [ ]:
result = run_evaluation(config)
result


## 5. Package the results

In [ ]:
import shutil
shutil.make_archive(f"/kaggle/working/backbone_eval_{config.fingerprint()}",
                    "zip", config.output_root)
